# MCTS-over-Landmarks E1 Runner

Notebook Kaggle này clone đúng nhánh `feature/mcts-landmark`, cài dependencies, rồi chạy E1a/E1b/E1c/E1d qua `scripts/kaggle_run_e1_all.py`.

Cách dùng nhanh:
- Set `HF_REPO_ID` để tải checkpoint `.pt` từ Hugging Face Dataset repo, hoặc add Kaggle Dataset chứa checkpoint.
- Chọn `PRESET = "smoke"` để test pipeline, `"report"` để chạy nghiêm túc hơn.
- Với env MuJoCo (`PointMazeMuJoCo`, `FetchPickAndPlace`, `AntMaze`), bật `INSTALL_MUJOCO = True`.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

REPO_URL = "https://github.com/Jun1801/latent_landmarks.git"
BRANCH = "feature/mcts-landmark"
REPO_DIR = Path("/kaggle/working/latent_landmarks")

# Presets: smoke | report | full
PRESET = "smoke"

# Optional: restrict tasks/experiments for faster runs.
# Examples: ONLY = ["pointmaze_numpy"] ; EXPERIMENTS = ["e1a", "e1b"]
ONLY = []
EXPERIMENTS = []

# Hugging Face Dataset repo chứa checkpoint .pt.
# Để trống nếu bạn dùng Kaggle Dataset hoặc checkpoint đã có trong repo clone.
HF_REPO_ID = "Jun1801/mcts_vla"
DOWNLOAD_CHECKPOINTS_FROM_HF = bool(HF_REPO_ID)

# Optional custom manifest. Leave as None to use the runner defaults.
MANIFEST = None

# MuJoCo install is only needed for PointMazeMuJoCo / Fetch / AntMaze.
INSTALL_MUJOCO = False

OUTPUT_ROOT = Path("/kaggle/working/e1_runs")

## Clone Repo

Cell này clone repo bằng `--single-branch --branch feature/mcts-landmark`. Nếu thư mục đã tồn tại, nó checkout nhánh đó và pull fast-forward.

In [ ]:
if not REPO_DIR.exists():
    subprocess.run([
        "git", "clone", "--branch", BRANCH, "--single-branch",
        REPO_URL, str(REPO_DIR)
    ], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)

os.chdir(REPO_DIR)
print("cwd:", Path.cwd())
subprocess.run(["git", "branch", "--show-current"], check=True)
subprocess.run(["git", "rev-parse", "--short", "HEAD"], check=True)

## Install Dependencies

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=True)

if INSTALL_MUJOCO:
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "mujoco>=3", "gymnasium-robotics"
    ], check=True)

print("Dependencies ready.")

## Download Checkpoints From Hugging Face

Nếu `HF_REPO_ID` được set, cell này kéo toàn bộ `*.pt` từ Hugging Face Dataset repo về `checkpoint/`.

In [ ]:
if DOWNLOAD_CHECKPOINTS_FROM_HF:
    cmd = [
        sys.executable, "scripts/hf_sync_checkpoints.py", "download",
        "--repo-id", HF_REPO_ID,
        "--checkpoint-dir", "checkpoint",
    ]
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    print("HF checkpoint download skipped. Set HF_REPO_ID to enable it.")

pts = sorted(Path("checkpoint").rglob("*.pt"))
print(f"Found {len(pts)} local checkpoint(s) under checkpoint/")
for p in pts[:50]:
    print(p)

## Inspect Kaggle Inputs

Runner tự tìm checkpoint theo tên trong repo và dưới `/kaggle/input`. Cell này chỉ để kiểm tra nhanh những file `.pt` đang có.

In [ ]:
input_root = Path("/kaggle/input")
if input_root.exists():
    pts = sorted(input_root.rglob("*.pt"))
    print(f"Found {len(pts)} checkpoint(s) under /kaggle/input")
    for p in pts[:50]:
        print(p)
else:
    print("/kaggle/input not found; runner will use checkpoints in the cloned repo if present.")

## Optional Custom Manifest

Nếu muốn chạy trên bộ checkpoint/dataset khác, sửa cell dưới và set `MANIFEST = str(custom_manifest)`.

In [ ]:
custom_manifest = Path("/kaggle/working/e1_tasks.json")

# Uncomment và sửa checkpoint path nếu cần.
# custom_tasks = {
#     "tasks": [
#         {
#             "name": "pointmaze_alt_seed",
#             "env": "PointMaze",
#             "checkpoint": "/kaggle/input/your-dataset/l3p_pointmaze_seed1.pt",
#             "experiments": ["e1a", "e1b", "e1c", "e1d"],
#         },
#         {
#             "name": "fetch_pick_and_place",
#             "env": "FetchPickAndPlace",
#             "checkpoint": "/kaggle/input/your-dataset/l3p_fetch.pt",
#             "experiments": ["e1a", "e1b"],
#         },
#     ]
# }
# custom_manifest.write_text(json.dumps(custom_tasks, indent=2))
# MANIFEST = str(custom_manifest)

print("MANIFEST =", MANIFEST)

## Dry Run

In command sẽ chạy, chưa chạy experiment thật.

In [ ]:
cmd = [
    sys.executable, "scripts/kaggle_run_e1_all.py",
    "--preset", PRESET,
    "--output-root", str(OUTPUT_ROOT),
    "--dry-run",
]
if MANIFEST:
    cmd += ["--manifest", MANIFEST]
if ONLY:
    cmd += ["--only", *ONLY]
if EXPERIMENTS:
    cmd += ["--experiments", *EXPERIMENTS]

print("$", " ".join(map(str, cmd)))
subprocess.run(cmd, check=True)

## Run E1 Batch

In [ ]:
cmd = [
    sys.executable, "scripts/kaggle_run_e1_all.py",
    "--preset", PRESET,
    "--output-root", str(OUTPUT_ROOT),
]
if MANIFEST:
    cmd += ["--manifest", MANIFEST]
if ONLY:
    cmd += ["--only", *ONLY]
if EXPERIMENTS:
    cmd += ["--experiments", *EXPERIMENTS]

print("$", " ".join(map(str, cmd)))
subprocess.run(cmd, check=True)

## Plot Trajectory

Cell này vẽ trajectory cho checkpoint PointMaze mặc định sau khi batch chạy xong. Với MuJoCo env, dùng `scripts/plot_trajectory_mujoco.py` và checkpoint/env tương ứng.

In [ ]:
traj_out = OUTPUT_ROOT / "pointmaze_numpy" / "trajectory.png"
traj_out.parent.mkdir(parents=True, exist_ok=True)
mpl_dir = OUTPUT_ROOT / "mplconfig"
mpl_dir.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(mpl_dir))

subprocess.run([
    sys.executable, "scripts/plot_trajectory.py",
    "--load", "checkpoint/l3p_pointmaze_full.pt",
    "--out", str(traj_out),
], check=True)

print("Saved trajectory plot to", traj_out)

## Display Plots

Hiển thị tất cả PNG đã sinh ra: E1 curves và trajectory.

In [ ]:
pngs = sorted(OUTPUT_ROOT.rglob("*.png"))
print(f"Found {len(pngs)} PNG plot(s).")
try:
    from IPython.display import Image, display
except ImportError:
    Image = display = None
    print("IPython is not available; listing plot paths only.")

for p in pngs:
    print(p)
    if Image is not None:
        display(Image(filename=str(p)))

## Summary

In [ ]:
summary_path = OUTPUT_ROOT / "summary.json"
print("summary:", summary_path)
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print(json.dumps(summary, indent=2)[:8000])
else:
    print("No summary yet. Run the E1 batch cell first.")